# 3-qubit error correction

## Introduction

The QEC3 algorithm is an error correction algorithm that protects against a single bit-flip error, it encodes 1 logical qubit into 3 physical qubits.

In this exercise, we will use the QEC3 algorithm to:
- Implement an hybrid circuit with a different gate set
- Evaluate the circuit into an HPS and rewrite using the HH rule
- Concretize the HPS into a Vector Map and compute some probabilities
- Verify the probability of valid output

## Circuit

![QEC3 circuit](../../img/qec3.png)

**Notes**:
- $I$ is a gate that does a bit flip with a probability p, and do nothing otherwise
- The white controls are controls of the form: if **not** $c_0$ then ...
- The double controls are controls of the form if $c_0$ **and** $c_1$ then ...

## Circuit implementation

Implement a function that generates a qec3 circuit from a probability p of error.  
We are gonna use a different gate set **Clifford_k_err** containing the same gates as usual with the addition of an error gate called **Ie**. This gate takes a scalar of the form **Hps.Scalar.(SFrac (Z.of_int 1, Z.of_int 1O))** as parameter, which represents the probability of introducing an error.  
There is an **and -> &** and a **not -> !** operator for the controls at the end.

In [ ]:
#require "hqbricks"
open Hqbricks
open Gate_set_impl.Clifford_k_err

let generate_qec3 p =
  let open Prog in
  let psi = qreg "psi" ~$1 in
  let q = qreg "q" ~$2 in
  let q0 = q_idx q ~$0 in
  let q1 = q_idx q ~$1 in
  let c = qreg "c" ~$2 in
  let mc = creg "mc" ~$2 in
  let b_mc0 = cbit_val mc ~$0 in
  let b_mc1 = cbit_val mc ~$1 in
  init_qreg q
  -- (qbit_val psi ~$0 => (q0 |> x))
  -- (qbit_val q ~$0 => (q1 |> x))
  -- (psi |> ie p)
  -- (q |> ie p)
  -- init_qreg c
  -- (c |> h)
  -- (qbit_val c ~$0 => (psi |> z)
  -- (q0 |> z))
  -- (qbit_val c ~$1 => (q0 |> z)
  -- (q1 |> z))
  -- (c |> h)
  -- (c -@ mc)
  -- ((b_mc0 & b_mc1) => (q0 |> x))
  -- ((b_mc0 & !b_mc1) => (psi |> x))
  -- ((!b_mc0 & b_mc1) => (q1 |> x))

Generate qec3 with $p = \frac{2}{3}$:

In [ ]:
let p = Hps.Scalar.(SFrac (Z.of_int 2, Z.of_int 3))
let qec3_circ = generate_qec3 p
let () =
  print_endline "\nCirc:";
  Prog.print qec3_circ

## HPS evaluation and rewrite

First, let's look at the evaluated HPS without any rewrite:

In [ ]:
let input_hps = Hps.add_qmem_vec_x ("psi", 0) 1 0 Hps.one
let () =
  print_endline "\nInput HPS:";
  print_endline (Hps.to_string input_hps);
  print_endline ""

let hps = Evaluator.(evaluate_prog qec3_circ input_hps ~print:false)
let () =
  print_endline "\nHPS:";
  print_endline (Hps.to_string hps)

We will now try to reduce the number of $y$ path variables by using the HH rewrite rule.

**Exercise**:  
We will try to evaluate the qec3 circuit with interactive rewrite and verify the resulting HPS.
Open a terminal by clicking File -> New -> Terminal then run `cd examples/qec3 && dune build && cd _build/default/bin` and `./main.exe`. You can then see the evaluation step by step (press enter to go to next step) while being able to run rewrite commands at each step.  
The goal of this exercise is to validate the verification by using **hh** command:
- `hh y0 y1` to try to apply hh on y0 and y1, displays why if it cannot
- `hh -find` to find the current possible HH
- `hh -all` to find and apply all current possible HH
- As you will see, hh has several conditions of application
Tip: one of the conditions is that both y must not appear in the classical memory, so the hh must be done before the measures.

Now evaluate the HPS with **automatic rewrite**:

In [ ]:
let hps = Evaluator.(evaluate_prog qec3_circ input_hps ~rewrite_settings:all_auto)
let () =
  print_endline "\nHPS:";
  print_endline (Hps.to_string hps)

## Vector Map and Probability concretization

Concretize the HPS into a vector map:

In [ ]:
open Concretization

let vm = Vector_map.of_hps hps
let () =
  print_endline "\nVector Map:";
  print_endline (Vector_map.to_string vm)

Since there are 3 $y$ path variables, there are 8 possible paths (1 for each possible combination of values of the $y$), and we also have 8 worlds (8 distinct possible classical memory), so in this case we have 1 world per path, meaning that there was no interference.  
Here, we can compute the probability of having no error $\ket{x_0}_\psi$ $\ket{x_0x_0}_q$ with the **hps_proba_qmem** function with a qmem spec $\ket{x_0}_\psi$ $\ket{x_0x_0}_q$ as parameter:

In [ ]:
let x0 = Hps.Hket.of_var @@ X 0
let spec_qmem = Hps.Mem.(Stdlib.(empty |> add ("psi", 0) x0 |> add ("q", 0) x0 |> add ("q", 1) x0))
let () =
  print_endline "\nQmem Spec:";
  print_endline (Hps.Mem.qmem_to_string spec_qmem)

let success_proba = hps_proba_qmem spec_qmem hps
let () =
  print_endline "\nSuccess Proba:";
  print_endline (Hps.Scalar.to_string success_proba)

## Verification

Knowing the success probability in advance, we can verify the success probability of the hps with:

In [ ]:
let exp_proba = Hps.Scalar.(SFrac (Z.of_int 7, Z.of_int 27)) in
assert (Assertion.hps_satisfies_proba_qmem exp_proba spec_qmem hps)

You may have noticed that for $p = \frac{2}{3}$ the probability of success of QEC3 is lesser than the initial probability of success $\frac{1}{3}$, but the QEC3 becomes better and better as $p$ decreases. For example for $p = \frac{1}{3}$ QEC3 have a probability of success of $\frac{20}{27}$ which is better than $1 - p$.  
Can you find for which value of $p$ the probability of success of QEC3 becomes better than $1 - p$ ?

Equal for $p = \frac{1}{2}$, better after